# Amazon Bedrock AgentCore Forecasting Agent using Amazon Chronos

We start by importing the required libraries: `boto3` to invoke the agent on Amazon Bedrock AgentCore, `json` to serialize the request payload and deserialize the response, `uuid` to generate unique session identifiers, and `IPython.display` to render the agent's responses as formatted `Markdown` in the notebook.

In [1]:
import json
import boto3
import uuid
from IPython.display import Markdown, display

Next, we set up the client and helper functions. We define the AgentCore Runtime ARN and region, create a `boto3` client for invoking the agent, and implement three helper functions: `parse_streaming_response` to extract agent messages from the response stream, `get_streaming_response` to invoke the agent and return the parsed messages, and `print_messages` to render a conversation turn in the notebook with formatted text, tool calls and tool results.

In [2]:
# Configure the Bedrock AgentCore runtime
AGENTCORE_RUNTIME_ARN = "<agentcore-runtime-arn>"
REGION = "<agentcore-runtime-region>"

# Create the Bedrock AgentCore client
agentcore_client = boto3.client(
    service_name="bedrock-agentcore",
    region_name=REGION
)

def parse_streaming_response(response: dict) -> list[dict]:
    """
    Parse the streaming response from AgentCore and extract agent messages.

    Parameters:
    ===============================================================================
    response: dict
        The raw response dictionary returned by invoke_agent_runtime,
        containing a StreamingBody object under the "response" key.

    Returns:
    ===============================================================================
    list[dict]
        List of agent message dictionaries extracted from the event stream.
    """
    messages = []
    for line in response["response"].iter_lines():
        if line:
            data = line.decode("utf-8")
            try:
                start, end = data.index("{"), data.rindex("}")
                event = json.loads(data[start:end + 1], strict=False)
                if "message" in event:
                    messages.append(event["message"])
            except (ValueError, json.JSONDecodeError):
                pass
    return messages


def get_streaming_response(prompt: str, session_id: str) -> list[dict]:
    """
    Invoke the forecasting agent on Amazon Bedrock AgentCore and return
    the agent messages.

    Parameters:
    ===============================================================================
    prompt: str
        The user message to send to the agent.

    session_id: str
        The session identifier used to maintain conversation context across turns.

    Returns:
    ===============================================================================
    list[dict]
        List of agent message dictionaries.
    """
    # Invoke the agent on Bedrock AgentCore
    response = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=AGENTCORE_RUNTIME_ARN,
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}),
    )
    
    # Parse and return the streaming response
    messages = parse_streaming_response(response)
    return messages


def print_messages(prompt: str, messages: list[dict]) -> None:
    """
    Display a conversation turn in a Jupyter notebook, including the user
    prompt, agent text responses, tool calls and tool results.

    Parameters:
    ===============================================================================
    prompt: str
        The user message sent to the agent.

    messages: list[dict]
        List of agent message dictionaries returned by the agent.
    """
    # Display the user prompt
    display(Markdown(f"<h2>User:</h2><br>{prompt}"))

    # Display the agent response
    display(Markdown("<h2>Agent:</h2><br>"))
    for message in messages:
        for content in message["content"]:
            if "text" in content:
                # Display text response as formatted markdown
                display(Markdown(content["text"]))
            elif "toolUse" in content:
                # Display tool call name and inputs
                display(Markdown(f"🔨 Ran `{content['toolUse']['name']}`"))
                print({"input": content["toolUse"]["input"]})
            elif "toolResult" in content:
                # Display tool result output
                if "content" in content["toolResult"]:
                    tool_output = content["toolResult"]["content"][0]["text"]
                    display(Markdown("🔨 Output:"))
                    print({"output": json.loads(tool_output)})

We generate a unique session ID to identify this conversation. The same session ID will be reused across all turns to maintain conversation context and short-term memory.

In [3]:
# Generate the session ID
session_id = str(uuid.uuid4())

We start the conversation by asking the agent what it can do.

In [4]:
prompt = """
What can you help me with?
"""
messages = get_streaming_response(prompt, session_id)
print_messages(prompt, messages)

<h2>User:</h2><br>
What can you help me with?


<h2>Agent:</h2><br>

I'm a **time series forecasting assistant**! Here's what I can help you with:

---

### 📈 What I Do
I generate **probabilistic time series forecasts** from your historical numerical data. This means I don't just predict a single value — I provide a **range of possible future outcomes** based on uncertainty.

---

### 🔢 What I Need From You
To generate a forecast, I'll need:

1. **Historical Data** — A list of numerical values representing your time series (e.g., daily sales, hourly temperatures, monthly revenue).
2. **Prediction Length** — How many future time steps you want to forecast (e.g., 7 for the next 7 days).
3. **Quantile Levels** — The probability levels for uncertainty bounds (e.g., `[0.1, 0.5, 0.9]` gives you a low, median, and high estimate).

---

### 💡 Example Use Cases
- 📦 **Demand forecasting** — Predict future product sales
- 🌡️ **Weather trends** — Forecast temperature or rainfall
- 💰 **Financial data** — Project revenue or stock trends
- ⚡ **Energy consumption** — Anticipate future usage
- 🌐 **Web traffic** — Estimate future site visits

---

### 🚀 Getting Started
Simply share your **time series data**, and let me know:
- How far ahead you'd like to forecast
- What quantile levels you want

I'll take care of the rest!

In the second turn, we provide a time series and ask the agent to forecast the next 10 values. 

In [5]:
prompt = """
Can we predict the next 10 values of this time series? 
[16, 6, 0, 3, 13, 20, 18, 9, 1, 1, 10, 18, 19, 12, 3, 0, 7, 16, 20, 15, 5, 0, 4, 14, 20, 17, 8, 1, 2, 11, 19, 19, 11, 2, 1, 8, 17, 20, 14, 4, 0, 5, 14, 20, 16, 7, 0, 3, 12, 19, 18, 10, 2, 1, 9, 18, 20, 13, 3, 0, 6, 15, 20, 16]
"""
messages = get_streaming_response(prompt, session_id)
print_messages(prompt, messages)

<h2>User:</h2><br>
Can we predict the next 10 values of this time series? 
[16, 6, 0, 3, 13, 20, 18, 9, 1, 1, 10, 18, 19, 12, 3, 0, 7, 16, 20, 15, 5, 0, 4, 14, 20, 17, 8, 1, 2, 11, 19, 19, 11, 2, 1, 8, 17, 20, 14, 4, 0, 5, 14, 20, 16, 7, 0, 3, 12, 19, 18, 10, 2, 1, 9, 18, 20, 13, 3, 0, 6, 15, 20, 16]


<h2>Agent:</h2><br>

Sure! Before I generate the forecast, I just need one more detail:

**What quantile levels would you like?**
These represent the uncertainty bounds of the forecast. For example:
- `[0.1, 0.5, 0.9]` → low, median, and high estimates
- `[0.25, 0.5, 0.75]` → tighter range around the median

What would you prefer?

In the third turn, we provide the requested quantile levels and specify that we want the mean instead of the median. The agent now has all the information it needs and calls `generate_forecasts`, returning the forecast values and the 95% prediction interval.

In [6]:
prompt = """
I need a 95% prediction interval. I don't need the median, only the mean.
"""
messages = get_streaming_response(prompt, session_id)
print_messages(prompt, messages)

<h2>User:</h2><br>
I need a 95% prediction interval. I don't need the median, only the mean.


<h2>Agent:</h2><br>

Got it! I'll use quantile levels `[0.025, 0.975]` for the 95% prediction interval, along with the mean.

🔨 Ran `generate_forecasts`

{'input': {'target': [16, 6, 0, 3, 13, 20, 18, 9, 1, 1, 10, 18, 19, 12, 3, 0, 7, 16, 20, 15, 5, 0, 4, 14, 20, 17, 8, 1, 2, 11, 19, 19, 11, 2, 1, 8, 17, 20, 14, 4, 0, 5, 14, 20, 16, 7, 0, 3, 12, 19, 18, 10, 2, 1, 9, 18, 20, 13, 3, 0, 6, 15, 20, 16], 'prediction_length': 10, 'quantile_levels': [0.025, 0.975]}}


🔨 Output:

{'output': {'mean': [7.842952728271484, 3.9406585693359375, 4.136490345001221, 7.826767921447754, 15.442928314208984, 15.398470878601074, 9.315390586853027, 5.722835063934326, 5.371423244476318, 7.833820343017578], '0.025': [2.4480767250061035, -0.8402576446533203, -0.2643604278564453, 2.227060317993164, 7.496842384338379, 7.033439636230469, 2.4833216667175293, -0.3372507095336914, -0.5436820983886719, 0.8084659576416016], '0.975': [13.956232070922852, 10.066437721252441, 10.064048767089844, 14.887125015258789, 23.994970321655273, 24.602264404296875, 17.92835235595703, 13.450220108032227, 13.074174880981445, 16.784805297851562]}}


Here are the forecast results for the next 10 time steps:

| Step | Mean | Lower 2.5% | Upper 97.5% |
|------|------|------------|-------------|
| 1  | 7.84  | 2.45  | 13.96 |
| 2  | 3.94  | -0.84 | 10.07 |
| 3  | 4.14  | -0.26 | 10.06 |
| 4  | 7.83  | 2.23  | 14.89 |
| 5  | 15.44 | 7.50  | 23.99 |
| 6  | 15.40 | 7.03  | 24.60 |
| 7  | 9.32  | 2.48  | 17.93 |
| 8  | 5.72  | -0.34 | 13.45 |
| 9  | 5.37  | -0.54 | 13.07 |
| 10 | 7.83  | 0.81  | 16.78 |

---

### 🔍 Key Observations:
- **Cyclical pattern detected** — The model has picked up on the wave-like pattern in your data (which oscillates between ~0 and ~20), as seen in the predicted means rising and falling across the 10 steps.
- **95% Prediction Interval** — The lower bound dips slightly below 0 for some steps, which is consistent with the uncertainty in the model around the troughs of the cycle.
- **Peak around steps 5–6** — The forecast anticipates a peak (~15.4) mid-way, aligning with the recurring cycle in the historical data.